In [1]:
import sqlite3
import pandas as pd

In [3]:
conn = sqlite3.connect("olist.db")

# 
# Load core tables
orders = pd.read_sql("SELECT * FROM dim_orders;", conn)
order_items = pd.read_sql("SELECT * FROM fact_order_items;", conn)
customers = pd.read_sql("SELECT * FROM dim_customers;", conn)
products = pd.read_sql("SELECT * FROM dim_products;", conn)
payments = pd.read_sql("SELECT * FROM fact_payments;", conn)
reviews = pd.read_sql("SELECT * FROM fact_reviews;", conn)
sellers = pd.read_sql("SELECT * FROM dim_sellers;", conn)

In [5]:
date_cols = [
    "order_purchase_timestamp", 
    "order_delivered_customer_date", 
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Quick null check
orders.isnull().sum()


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [7]:
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')


In [9]:
orders['delivery_delay'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days


In [11]:
order_items['order_value'] = order_items['price'] + order_items['freight_value']


In [13]:
customer_spend = (order_items
                  .merge(orders[['order_id','customer_id']], on='order_id')
                  .groupby('customer_id')['order_value']
                  .sum()
                  .reset_index()
                  .rename(columns={'order_value': 'customer_lifetime_value'}))


In [15]:
customer_freq = (orders.groupby('customer_id')['order_id']
                        .count()
                        .reset_index()
                        .rename(columns={'order_id': 'num_orders'}))


# aggregation

In [18]:
# Monthly GMV
monthly_gmv = (order_items
               .merge(orders[['order_id', 'order_month']], on='order_id')
               .groupby('order_month')['order_value']
               .sum()
               .reset_index())

# Top categories
top_categories = (order_items
                  .merge(products[['product_id', 'product_category_name']], on='product_id')
                  .groupby('product_category_name')['order_value']
                  .sum()
                  .reset_index()
                  .sort_values(by='order_value', ascending=False)
                  .head(10))


# saving processed data

In [23]:
monthly_gmv.to_csv("./monthly_gmv.csv", index=False)
top_categories.to_csv("./top_categories.csv", index=False)
customer_spend.to_csv("./customer_lifetime_value.csv", index=False)
customer_freq.to_csv("./customer_frequency.csv", index=False)
